**Đọc dataset**

In [1]:
!pip install -q kagglehub[pandas-datasets]

import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd

file_path = "go_emotions_dataset.csv"


df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "shivamb/go-emotions-google-emotions-dataset",
    file_path
)

# Khai báo cấu trúc hệ thống phân loại 28 nhãn chuẩn của Google
emotion_labels = [
    "admiration", "amusement", "anger", "annoyance", "approval", "caring",
    "confusion", "curiosity", "desire", "disappointment", "disapproval",
    "disgust", "embarrassment", "excitement", "fear", "gratitude", "grief",
    "joy", "love", "nervousness", "optimism", "pride", "realization",
    "relief", "remorse", "sadness", "surprise", "neutral"
]



/tmp/ipykernel_9467/2057014050.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'go-emotions-google-emotions-dataset' dataset.


**Gom cụm**

In [2]:
# Gom cụm các rater đánh giá cùng một câu bình luận lại với nhau
grouped = df.groupby(['id', 'text'])[emotion_labels].sum().reset_index()
print(f"Số lượng câu bình luận độc nhất sau khi gom cụm: {len(grouped)} câu.")

# Nhãn cảm xúc nào được ít nhất 2 chuyên gia chọn mới tính là 1, dưới 2 phiếu bầu sẽ bị hủy (= 0)
for col in emotion_labels:
    grouped[col] = (grouped[col] >= 2).astype(int)

# Tính tổng số nhãn được kích hoạt cho mỗi câu
grouped['has_label'] = grouped[emotion_labels].sum(axis=1)
# Chỉ giữ lại các câu có ít nhất 1 nhãn cảm xúc đạt sự đồng thuận từ 2 người trở lên
grouped_clean = grouped[grouped['has_label'] > 0].drop(columns=['has_label'])

print(f"🔹 Số lượng câu bình luận sạch thu được: {len(grouped_clean)} câu.")

# Xem thử 3 dòng đầu tiên sau khi làm sạch
grouped_clean.head(3)

Số lượng câu bình luận độc nhất sau khi gom cụm: 58011 câu.
🔹 Số lượng câu bình luận sạch thu được: 54263 câu.


,id,text,admiration,amusement,anger,annoyance,approval,caring,confusion,curiosity,...,love,nervousness,optimism,pride,realization,relief,remorse,sadness,surprise,neutral
0,eczazk6,Fast as [NAME] will carry me. Seriously uptown...,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,eczb07q,You blew it. They played you like a fiddle.,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,1
2,eczb4bm,TL;DR No more Superbowls for [NAME]. Get ready...,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


**Làm sạch**

In [3]:
import re
import unicodedata
import nltk
from nltk.stem import WordNetLemmatizer

# Tải thư viện cần thiết (thêm quiet=True để giảm log rác trên màn hình)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

lemmatizer = WordNetLemmatizer()

def clean_text_for_cnn(text, use_lemmatization=False):
    # 1. Tránh lỗi crash nếu dữ liệu bị trống (NaN)
    if not isinstance(text, str):
        return ""

    # 2. Đưa về chữ thường
    text = text.lower()

    # 3. Đồng bộ hóa "dấu nháy" và xử lý Unicode
    text = re.sub(r"[’‘`´]", "'", text)
    text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

    # 4. Khai triển từ viết tắt (RẤT QUAN TRỌNG để Word2Vec không bị trượt từ)
    contractions = {
        r"i'm": "i am", r"don't": "do not", r"doesn't": "does not",
        r"can't": "cannot", r"won't": "will not", r"isn't": "is not",
        r"aren't": "are not", r"haven't": "have not", r"hasn't": "has not",
        r"didn't": "did not", r"it's": "it is", r"that's": "that is",
        r"you're": "you are", r"i've": "i have", r"i'll": "i will"
    }
    for pattern, replacement in contractions.items():
        text = re.sub(pattern, replacement, text)

    # Vớt những cụm viết tắt đuôi còn sót
    text = re.sub(r"n't", " not", text)
    text = re.sub(r"'re", " are", text)
    text = re.sub(r"'s", " is", text)
    text = re.sub(r"'d", " would", text)
    text = re.sub(r"'ll", " will", text)
    text = re.sub(r"'ve", " have", text)
    text = re.sub(r"'m", " am", text)

    # ĐÃ LOẠI BỎ: Kỹ thuật nối từ phủ định (not_word) để CNN tự học thông qua Kernel

    # 5. Lọc ký tự đặc biệt (giữ lại khoảng trắng, a-z, 0-9 và dấu biểu cảm ! ? _)
    text = re.sub(r"[^a-z0-9\s\!\?\_]", " ", text)

    # 6. Ép các ký tự lặp lại vô nghĩa (soooo -> so)
    text = re.sub(r'(.)\1{2,}', r'\1', text)

    # 7. Gom khoảng trắng thừa và cắt tỉa
    text = re.sub(r"\s+", " ", text).strip()

    # 8. TÙY CHỌN: Lemmatization (Ép gốc từ)
    if use_lemmatization:
        words = text.split()
        # pos='v' giúp ép các động từ về nguyên thể (love, loved, loving -> love)
        text = " ".join([lemmatizer.lemmatize(w, pos='v') for w in words])

    return text

# ========================================================
# ÁP DỤNG HÀM LÀM SẠCH LÊN TẬP DỮ LIỆU
# ========================================================

# Tạo bản sao để tránh lỗi SettingWithCopyWarning của Pandas
df_ready = grouped_clean.copy()

# Xóa các dòng rỗng hoàn toàn trước khi xử lý
df_ready = df_ready.dropna(subset=['text'])

# Gọi hàm làm sạch.
# GHI CHÚ: Đặt use_lemmatization=False nếu bạn dùng mô hình Word2Vec tải trên mạng (Pre-trained)
# Đặt use_lemmatization=True nếu bạn tự code thuật toán train Word2Vec từ đầu (Train from scratch)
df_ready['text'] = df_ready['text'].apply(lambda x: clean_text_for_cnn(x, use_lemmatization=False))

# Loại bỏ những câu mà sau khi làm sạch chỉ còn là chuỗi rỗng
df_ready = df_ready[df_ready['text'] != ""]

**Chia tập train valid test**

In [4]:
from sklearn.model_selection import train_test_split

# 1. Tách ma trận đầu vào X (văn bản SẠCH) và ma trận đầu ra Y (28 nhãn)
X_text = df_ready['text'].astype(str).values      # <--- Sửa thành df_ready
Y_matrix = df_ready[emotion_labels].values        # <--- Sửa thành df_ready

# 2. Lần tách thứ nhất: Giữ lại 80% cho tập Train, đẩy 20% còn lại vào tập tạm (Temp)
X_train_text, X_temp_text, Y_train, Y_temp = train_test_split(
    X_text, Y_matrix, test_size=0.20, random_state=42
)

# 3. Lần tách thứ hai: Chia đôi 50/50 tập tạm (Temp) để lấy ra đúng 10% Val và 10% Test
X_val_text, X_test_text, Y_val, Y_test = train_test_split(
    X_temp_text, Y_temp, test_size=0.50, random_state=42
)

print(f"Tập Huấn luyện (Train - 80%): {X_train_text.shape[0]} câu.")
print(f"Tập Phát triển/Kiểm định (Val - 10%): {X_val_text.shape[0]} câu.")
print(f"Tập Kiểm thử (Test - 10%): {X_test_text.shape[0]} câu.")
print(f"Tổng cộng: {X_train_text.shape[0] + X_val_text.shape[0] + X_test_text.shape[0]} câu.")

Tập Huấn luyện (Train - 80%): 43408 câu.
Tập Phát triển/Kiểm định (Val - 10%): 5426 câu.
Tập Kiểm thử (Test - 10%): 5426 câu.
Tổng cộng: 54260 câu.


**Tiền trạm (Tokenization & Padding)**

In [5]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np

# 1. Khởi tạo Tokenizer
# Giới hạn bộ từ vựng ở 20,000 từ phổ biến nhất. Các từ hiếm gặp sẽ bị gộp chung thành nhãn <OOV>
MAX_VOCAB_SIZE = 20000
tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, oov_token="<OOV>")

# BẮT BUỘC: Chỉ fit Tokenizer trên tập Train
tokenizer.fit_on_texts(X_train_text)

# Xem thử có bao nhiêu từ duy nhất trong tập Train
word_index = tokenizer.word_index
print(f"Số lượng từ duy nhất trong từ điển: {len(word_index)}")

# 2. Chuyển đổi văn bản thành chuỗi các con số
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_val_seq = tokenizer.texts_to_sequences(X_val_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

# 3. Đệm chuỗi (Padding)
MAX_LEN = 50 # Giới hạn độ dài mỗi câu là 50 từ. (Có thể tăng/giảm tùy vào thống kê chiều dài câu trong EDA)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_val_pad = pad_sequences(X_val_seq, maxlen=MAX_LEN, padding='post', truncating='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding='post', truncating='post')

print("\n--- Kiểm tra kích thước ma trận sau Padding ---")
print(f"X_train_pad shape: {X_train_pad.shape}")
print(f"X_val_pad shape: {X_val_pad.shape}")
print(f"X_test_pad shape: {X_test_pad.shape}")

Số lượng từ duy nhất trong từ điển: 25909

--- Kiểm tra kích thước ma trận sau Padding ---
X_train_pad shape: (43408, 50)
X_val_pad shape: (5426, 50)
X_test_pad shape: (5426, 50)


**Word2Vec**

In [6]:
# 0. Cài đặt thư viện gensim nếu môi trường chưa có
!pip install -q gensim

from gensim.models import Word2Vec
import numpy as np

# 1. Chuẩn bị dữ liệu cho Gensim (Gensim yêu cầu đầu vào là một list các từ)
sentences_for_w2v = [str(sentence).split() for sentence in X_train_text]

# 2. Huấn luyện mô hình Word2Vec từ đầu
print("⏳ Đang huấn luyện Word2Vec...")
w2v_model = Word2Vec(
    sentences=sentences_for_w2v,
    vector_size=100, # Biểu diễn mỗi từ bằng một vector 100 chiều
    window=5,        # Cửa sổ ngữ cảnh: nhìn 5 từ trước và 5 từ sau
    min_count=2,     # Loại bỏ các từ quá hiếm (xuất hiện dưới 2 lần)
    workers=4        # Số luồng CPU chạy song song cho nhanh
)
print("✅ Huấn luyện Word2Vec thành công!")

# 3. Tạo Ma trận Nhúng (Embedding Matrix) để nạp vào CNN
EMBEDDING_DIM = 100
# Kích thước từ điển phải cộng thêm 1 để dành chỗ cho index 0 (padding)
vocab_size = len(word_index) + 1
embedding_matrix = np.zeros((vocab_size, EMBEDDING_DIM))

# Bơm trọng số (vector) từ mô hình Word2Vec vừa học vào Ma trận Nhúng
oov_count = 0
for word, i in word_index.items():
    if word in w2v_model.wv:
        # Nếu từ có trong Word2Vec, lấy vector của nó
        embedding_matrix[i] = w2v_model.wv[word]
    else:
        # Nếu không có (Out-Of-Vocabulary), nó sẽ giữ nguyên là vector [0, 0,..., 0]
        oov_count += 1

print(f"🔹 Kích thước Embedding Matrix: {embedding_matrix.shape}")
print(f"🔹 Số lượng từ không có vector (OOV): {oov_count} từ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 58.0 MB/s eta 0:00:00
⏳ Đang huấn luyện Word2Vec...
✅ Huấn luyện Word2Vec thành công!
🔹 Kích thước Embedding Matrix: (25910, 100)
🔹 Số lượng từ không có vector (OOV): 12881 từ


**xây dựng cấu trúc mô hình 1D CNN**

In [7]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, concatenate
from tensorflow.keras.models import Model

NUM_CLASSES = len(emotion_labels)

inputs = Input(shape=(MAX_LEN,), name='input_layer')

# ĐÃ SỬA THÀNH TRUE ĐỂ MÔ HÌNH THOÁT KHỎI ĐIỂM 0.25
embedding_layer = Embedding(
    input_dim=vocab_size,
    output_dim=EMBEDDING_DIM,
    weights=[embedding_matrix],
    input_length=MAX_LEN,
    trainable=True
)(inputs)

# Kiến trúc Multi-Kernel
conv_2 = Conv1D(filters=128, kernel_size=2, activation='relu')(embedding_layer)
pool_2 = GlobalMaxPooling1D()(conv_2)

conv_3 = Conv1D(filters=128, kernel_size=3, activation='relu')(embedding_layer)
pool_3 = GlobalMaxPooling1D()(conv_3)

conv_4 = Conv1D(filters=128, kernel_size=4, activation='relu')(embedding_layer)
pool_4 = GlobalMaxPooling1D()(conv_4)

merged_features = concatenate([pool_2, pool_3, pool_4], axis=1)

dense_1 = Dense(128, activation='relu')(merged_features)
dropout_1 = Dropout(0.4)(dense_1)
outputs = Dense(NUM_CLASSES, activation='sigmoid')(dropout_1)

model_optimized = Model(inputs=inputs, outputs=outputs)
model_optimized.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50)        │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 50, 100)   │  2,591,000 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 49, 128)   │     25,728 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 48, 128)   │     38,528 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 47, 128)   │     51,328 │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d[0][0]      │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_1[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_max_pooling… │ (None, 128)       │          0 │ conv1d_2[0][0]    │
│ (GlobalMaxPooling1… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 384)       │          0 │ global_max_pooli… │
│ (Concatenate)       │                   │            │ global_max_pooli… │
│                     │                   │            │ global_max_pooli… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 128)       │     49,280 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 128)       │          0 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 28)        │      3,612 │ dropout[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 2,759,476 (10.53 MB)

 Trainable params: 2,759,476 (10.53 MB)

 Non-trainable params: 0 (0.00 B)

**thiết lập biên dịch (Compile) và Huấn luyện (Training)**

In [8]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

model_optimized.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=[tf.keras.metrics.AUC(name='auc', multi_label=True)]
)

# Thêm "vũ khí mới": Tự động giảm Learning Rate nếu mô hình kẹt lại không học được tiếp
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-5, verbose=1)
early_stopping = EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True)

history = model_optimized.fit(
    X_train_pad, Y_train,
    epochs=20,
    batch_size=64,
    validation_data=(X_val_pad, Y_val),
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

Epoch 1/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 23s 17ms/step - auc: 0.6176 - loss: 0.1510 - val_auc: 0.7442 - val_loss: 0.1209 - learning_rate: 0.0010
Epoch 2/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 24s 6ms/step - auc: 0.7351 - loss: 0.1201 - val_auc: 0.7921 - val_loss: 0.1106 - learning_rate: 0.0010
Epoch 3/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - auc: 0.7976 - loss: 0.1087 - val_auc: 0.8221 - val_loss: 0.1044 - learning_rate: 0.0010
Epoch 4/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - auc: 0.8539 - loss: 0.0973 - val_auc: 0.8423 - val_loss: 0.1020 - learning_rate: 0.0010
Epoch 5/20
679/679 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - auc: 0.8880 - loss: 0.0879 - val_auc: 0.8456 - val_loss: 0.1053 - learning_rate: 0.0010
Epoch 6/20
669/679 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - auc: 0.9140 - loss: 0.0791
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
679/679 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - auc: 0.9139 - loss: 0.0798 - val_auc: 0.8390 - val_loss: 0.1099 - learning_rate: 0.0010

**Tìm ngưỡng tối ưu F1**

In [9]:
from sklearn.metrics import f1_score
import numpy as np

print("⏳ Đang dự đoán trên tập Validation...")
# LƯU Ý: Phải dùng tên model_optimized để dự đoán
Y_val_pred_probs = model_optimized.predict(X_val_pad)

best_thresholds = {}
f1_scores = {}

print("🔍 Bắt đầu dò tìm ngưỡng tối ưu cho từng nhãn...\n")
for i, label in enumerate(emotion_labels):
    best_thresh = 0.5
    best_f1 = 0.0

    for thresh in np.arange(0.01, 0.95, 0.01):
        y_pred_binary = (Y_val_pred_probs[:, i] >= thresh).astype(int)
        current_f1 = f1_score(Y_val[:, i], y_pred_binary, zero_division=0)

        if current_f1 > best_f1:
            best_f1 = current_f1
            best_thresh = thresh

    best_thresholds[label] = best_thresh
    f1_scores[label] = best_f1

    print(f"Nhãn: {label:>15} | Ngưỡng tối ưu: {best_thresh:.2f} | F1-Score: {best_f1:.4f}")

macro_f1 = np.mean(list(f1_scores.values()))
print(f"\n🏆 ĐIỂM MACRO F1 TRUNG BÌNH CỦA MÔ HÌNH: {macro_f1:.4f}")

⏳ Đang dự đoán trên tập Validation...
170/170 ━━━━━━━━━━━━━━━━━━━━ 3s 11ms/step
🔍 Bắt đầu dò tìm ngưỡng tối ưu cho từng nhãn...

Nhãn:      admiration | Ngưỡng tối ưu: 0.26 | F1-Score: 0.6756
Nhãn:       amusement | Ngưỡng tối ưu: 0.44 | F1-Score: 0.7864
Nhãn:           anger | Ngưỡng tối ưu: 0.20 | F1-Score: 0.4675
Nhãn:       annoyance | Ngưỡng tối ưu: 0.12 | F1-Score: 0.3085
Nhãn:        approval | Ngưỡng tối ưu: 0.11 | F1-Score: 0.2744
Nhãn:          caring | Ngưỡng tối ưu: 0.12 | F1-Score: 0.2770
Nhãn:       confusion | Ngưỡng tối ưu: 0.20 | F1-Score: 0.2907
Nhãn:       curiosity | Ngưỡng tối ưu: 0.22 | F1-Score: 0.4479
Nhãn:          desire | Ngưỡng tối ưu: 0.26 | F1-Score: 0.5096
Nhãn:  disappointment | Ngưỡng tối ưu: 0.11 | F1-Score: 0.1879
Nhãn:     disapproval | Ngưỡng tối ưu: 0.16 | F1-Score: 0.2993
Nhãn:         disgust | Ngưỡng tối ưu: 0.10 | F1-Score: 0.3819
Nhãn:   embarrassment | Ngưỡng tối ưu: 0.06 | F1-Score: 0.1613
Nhãn:      excitement | Ngưỡng tối ưu: 0.13 | F1-Sco

### **🚀 Triển khai Giao diện Demo với Gradio**
Đoạn mã dưới đây tạo một giao diện web cho phép bạn nhập văn bản và nhận dự đoán cảm xúc dựa trên mô hình CNN đã huấn luyện.

In [10]:
!pip install -q gradio

import gradio as gr
import numpy as np
from tensorflow.keras.preprocessing.sequence import pad_sequences

def predict_emotion(text):
    if not text.strip():
        return "Vui lòng nhập văn bản!"

    # 1. Làm sạch văn bản
    cleaned_text = clean_text_for_cnn(text, use_lemmatization=False)

    # 2. Tokenization & Padding
    seq = tokenizer.texts_to_sequences([cleaned_text])
    padded = pad_sequences(seq, maxlen=MAX_LEN, padding='post', truncating='post')

    # 3. Dự đoán xác suất từ model CNN
    probas = model_optimized.predict(padded)[0]

    # 4. Trình bày kết quả dựa trên bộ ngưỡng tối ưu
    results = {}
    activated = False

    for i, label in enumerate(emotion_labels):
        # Lấy ngưỡng đã tính toán trước đó cho nhãn này
        threshold_val = best_thresholds.get(label, 0.5)

        if probas[i] >= threshold_val:
            results[label] = float(probas[i])
            activated = True

    # Nếu không nhãn nào vượt ngưỡng, lấy nhãn có xác suất cao nhất
    if not activated:
        top_idx = np.argmax(probas)
        results[emotion_labels[top_idx]] = float(probas[top_idx])

    return results

# Khởi tạo giao diện Gradio
iface = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(lines=2, placeholder="Nhập câu tiếng Anh tại đây...", label="Input Text"),
    outputs=gr.Label(num_top_classes=5, label="Phân loại cảm xúc"),
    title="🚀 Google GoEmotions Predictor (CNN Edition)",
    description="Ứng dụng nhận diện 28 sắc thái cảm xúc dựa trên 1D CNN và bộ ngưỡng tối ưu F1.",
    examples=[
        ["I am so excited!"],
        ["This is disappointing."],
        ["Thank you for your help."],
        ["Wow, I am so surprised and happy for you!"],
        ["I love this, it is absolutely beautiful and amazing!"]
    ]
)

# Chạy giao diện
iface.launch(share=True, inline=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://017bf07b286eaffbfd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
